# Notebook 08: Unsupervised Anomaly Detection (Isolation Forest)
## ML-Based Liquidity and Trading-Risk Intelligence for Indian Equities

**Phase:** Phase 8 Batch 4 (Stages 8.9–8.11)
**Paradigm:** Unsupervised Anomaly Detection (Diagnostic Unusualness)
**Evaluation Protocol:** 10-Fold Expanding-Window Walk-Forward Validation

### Objectives
1. Fit `IsolationForest` on historical feature space using the selected 12-feature liquidity/risk subset.
2. Enforce strict architectural separation: Isolation Forest measures **historical distributional unusualness**, NOT forward risk.
3. Prevent lookahead leakage: fit strictly on post-warmup training observations up to the fold boundary.
4. Map raw decision function scores to rescaled 0–100 anomaly scores for dashboard consumption.
5. Cross-tabulate out-of-sample anomaly flags with forward High-Risk Events (`hre_5d_train`) to measure empirical overlap.


In [1]:
# Imports and Environment Setup
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add repository root to path regardless of working directory
current_dir = Path('.').resolve()
if (current_dir / 'data' / 'processed').exists():
    REPO_ROOT = current_dir
elif (current_dir.parent / 'data' / 'processed').exists():
    REPO_ROOT = current_dir.parent
else:
    REPO_ROOT = current_dir
    for p in [current_dir] + list(current_dir.parents):
        if (p / 'data' / 'processed').exists():
            REPO_ROOT = p
            break

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.modeling.anomaly import AnomalyDetector
from src.modeling.preprocessing import get_model_features

print(f'Repository root: {REPO_ROOT}')
print('Environment initialized successfully.')


Repository root: C:\Users\vighn\Downloads\Indian_Equities_Liquidity_Risk_ML-main\Indian_Equities_Liquidity_Risk_ML-main
Environment initialized successfully.


### 1. Feature Subset & Architecture

The Isolation Forest operates on a curated 12-feature subset capturing volume shocks, spread widening, and volatility bursts.
It is completely target-independent and includes observations even when forward labels are missing (e.g. trading halts).


In [2]:
if_features = get_model_features('isolation_forest')
print(f'Isolation Forest Features ({len(if_features)}):\n')
for i, f in enumerate(if_features, 1):
    print(f'  {i:2d}. {f}')

detector = AnomalyDetector(n_estimators=100, contamination=0.05, random_state=42)
print('\nAnomaly Detector configuration:', detector)


Isolation Forest Features (12):

   1. amihud_ratio
   2. amihud_z20
   3. volume_ratio
   4. volume_z20
   5. vol_20d
   6. vol_ratio_5_20
   7. daily_range
   8. range_z20
   9. overnight_gap_abs
  10. cs_spread_est
  11. nifty_vol_20d
  12. return_z20

Anomaly Detector configuration: <src.modeling.anomaly.AnomalyDetector object at 0x000001CBA8D65160>


### 2. Walk-Forward Out-of-Sample Anomaly Score Distributions

Inspect anomaly score metrics and flagging rates across all 10 walk-forward folds.


In [3]:
metrics_path = REPO_ROOT / 'data' / 'interim' / 'phase8_batch4' / 'isolation_forest_fold_metrics.csv'
if_metrics = pd.read_csv(metrics_path)

display_cols = [
    'fold', 'val_start', 'val_end', 'val_rows', 'mean_scaled_score',
    'median_scaled_score', 'p90_scaled_score', 'p95_scaled_score',
    'flagged_count', 'flagged_rate', 'hre_pos_flag_rate', 'anomaly_target_corr'
]
if_metrics[display_cols]


,fold,val_start,val_end,val_rows,mean_scaled_score,median_scaled_score,p90_scaled_score,p95_scaled_score,flagged_count,flagged_rate,hre_pos_flag_rate,anomaly_target_corr
0,0,2023-05-10,2023-08-02,15780,14.083335,11.447671,25.087342,34.470068,361,0.022877,0.026558,0.028228
1,1,2023-08-03,2023-10-31,15821,14.192683,11.683783,25.473851,33.654491,385,0.024335,0.031092,0.050647
2,2,2023-11-01,2024-01-30,15939,14.589745,11.350227,27.900975,37.783359,424,0.026601,0.031935,0.059952
3,3,2024-01-31,2024-04-30,16157,13.651991,10.085871,28.433404,38.743141,495,0.030637,0.051611,0.116106
4,4,2024-05-02,2024-07-29,16260,18.172469,14.931991,34.136427,43.306504,711,0.043727,0.051569,0.058905
5,5,2024-07-30,2024-10-23,16358,13.896194,10.952746,26.976472,35.281260,321,0.019623,0.023050,0.040811
6,6,2024-10-24,2025-01-20,16435,13.098585,9.761933,26.641475,35.149793,348,0.021174,0.037342,0.093789
7,7,2025-01-21,2025-04-21,16685,14.965709,11.575783,29.422224,39.090784,559,0.033503,0.046838,0.102743
8,8,2025-04-22,2025-07-15,16911,13.146722,10.456145,24.656040,32.351349,324,0.019159,0.022311,0.034017
9,9,2025-07-16,2025-10-10,17011,12.676896,9.998740,22.884354,31.558602,355,0.020869,0.018176,0.001085


### 3. Diagnostic Overlap with High-Risk Events (HRE)

Measure how frequently the unsupervised anomaly detector flags forward high-risk events.
Because the Isolation Forest is purely contemporaneous and unsupervised, empirical overlap confirms that liquidity stress manifests as distributional outliers.


In [4]:
print('=== Isolation Forest Summary Across 10 Folds ===')
print(f'Mean Scaled Score:    {if_metrics["mean_scaled_score"].mean():.2f} / 100')
print(f'Mean Flagged Rate:    {if_metrics["flagged_rate"].mean() * 100:.2f}% (Expected: ~5.00%)')
print(f'Mean HRE+ Flag Rate:  {if_metrics["hre_pos_flag_rate"].mean() * 100:.2f}%')
print(f'Mean HRE- Flag Rate:  {if_metrics["hre_neg_flag_rate"].mean() * 100:.2f}%')
print(f'Mean Correlation:     {if_metrics["anomaly_target_corr"].mean():+.4f}')


=== Isolation Forest Summary Across 10 Folds ===
Mean Scaled Score:    14.25 / 100
Mean Flagged Rate:    2.63% (Expected: ~5.00%)
Mean HRE+ Flag Rate:  3.40%
Mean HRE- Flag Rate:  2.45%
Mean Correlation:     +0.0586


### 4. Segment-Level Anomaly Prevalence

Evaluate anomaly scores and flag rates across Market Cap segments (Large, Mid, Small Cap).


In [5]:
preds_path = REPO_ROOT / 'data' / 'interim' / 'phase8_batch4' / 'isolation_forest_predictions.parquet'
if_preds = pd.read_parquet(preds_path)

cap_summary = if_preds.groupby('cap_segment').agg({
    'anomaly_score_scaled': ['count', 'mean', 'median', 'std'],
    'isolation_flag': ['mean', 'sum'],
    'y_true': ['mean']
}).round(4)
cap_summary


anomaly_score_scaled                            isolation_flag  \
                           count     mean   median      std           mean   
cap_segment                                                                  
large                      57011  13.5259  10.7618  10.0226         0.0196   
mid                        80292  14.0791  10.9982  10.7212         0.0254   
small                      26054  16.2771  12.5949  12.3992         0.0432   

                   y_true  
              sum    mean  
cap_segment                
large        1117  0.1829  
mid          2041  0.1860  
small        1125  0.1735